# PPE Detector Training -- YOLOv8n on Construction Safety Dataset

**Before running anything: go to `Runtime > Change runtime type` and set Hardware accelerator to `T4 GPU`.** Training on CPU here will be painfully slow; with a T4 a few thousand images at 50 epochs typically finishes in 20-50 minutes.

This notebook:
1. Installs `ultralytics` + dataset tooling
2. Downloads a labeled construction-PPE dataset (Option A: Roboflow, Option B: Kaggle mirror, no account needed)
3. Fine-tunes YOLOv8n (the smallest/fastest variant -- good for later CPU/edge inference)
4. Validates the model and runs sample predictions
5. Downloads `best.pt` to your laptop for the local webcam inference script

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import torch
print('GPU available:', torch.cuda.is_available())
print('If this prints False, go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## Step 1 -- Get the dataset

**Option A (recommended): Roboflow's "Construction Site Safety" dataset.**
It's already in YOLOv8 format with exactly the classes this project needs --
`Hardhat, NO-Hardhat, Safety Vest, NO-Safety Vest, Person, Mask, NO-Mask,
Safety Cone, machinery, vehicle` -- and has train/valid/test splits built in.

1. Go to https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety
2. Click **Download Dataset** (top right) -> format: **YOLOv8** -> **Show download code**
3. Roboflow gives you a code snippet with YOUR account's current API key and the
   dataset's current version number baked in -- **paste that exact snippet below**,
   replacing the placeholder cell. Don't reuse the version number from this notebook;
   it can go stale as the dataset gets updated.
4. Free Roboflow account required (roboflow.com/signup) -- no payment info needed.

**Option B: no Roboflow account needed.** Same dataset, mirrored on Kaggle
(https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow).
Use the Kaggle cell further down instead of the Roboflow cell.

In [ ]:
# --- OPTION A: Roboflow ---
# Replace this entire cell with the exact snippet Roboflow shows you after
# clicking "Show download code" on the dataset page -- it will look like this,
# but with YOUR real api_key and the CURRENT version number:

!pip install roboflow -q
from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(1)  # <-- confirm this matches the version shown on the dataset page
dataset = version.download("yolov8")

print('Downloaded to:', dataset.location)

In [ ]:
# --- OPTION B: Kaggle mirror (skip this cell if you used Option A above) ---
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download(
    "snehilsanyal/construction-site-safety-image-dataset-roboflow"
)
print('Downloaded to:', path)
# If prompted for Kaggle credentials: kaggle.com/settings -> Create New API Token
# -> upload the downloaded kaggle.json when Colab asks for it.

In [ ]:
# Locate data.yaml regardless of which option you used above --
# folder layouts can differ slightly between sources, so find it rather than
# assuming a path.
import glob

yaml_candidates = glob.glob('/content/**/data.yaml', recursive=True)
print('Found data.yaml at:')
for c in yaml_candidates:
    print(' ', c)

if yaml_candidates:
    DATA_YAML = yaml_candidates[0]
    print('\nUsing:', DATA_YAML)
    print('\n--- contents ---')
    with open(DATA_YAML) as f:
        print(f.read())
else:
    print('No data.yaml found yet -- run the download cell above first.')

## Step 2 -- Train

`yolov8n` = nano, the smallest variant. It trades a little accuracy for being
fast enough to run on a laptop CPU (and later a Raspberry Pi) without a GPU --
which is the whole point for the edge-deployment story. `epochs=50` and
`patience=10` (stop early if validation loss stalls for 10 epochs) are
reasonable starting points; raise epochs if your loss curve is still clearly
improving when training ends.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # starts from COCO-pretrained weights, then fine-tunes

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name='ppe_detector',
    patience=10,
)

## Step 3 -- Validate

mAP50 (mean average precision at IoU=0.5) is the headline number most teams
quote -- worth putting on your submission slide alongside what it means in
plain terms ("correctly identifies PPE violations in X% of held-out test
images").

In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print()
print('Per-class AP50:')
for i, name in model.names.items():
    try:
        print(f'  {name}: {metrics.box.ap50[i]:.3f}')
    except Exception:
        pass

In [ ]:
# Sanity-check on a few held-out validation images before trusting the model
import glob
from IPython.display import Image as IPImage, display

valid_images_dir = DATA_YAML.replace('data.yaml', 'valid/images')
sample_images = glob.glob(f'{valid_images_dir}/*.jpg')[:4]

pred_results = model.predict(sample_images, save=True, conf=0.4, verbose=False)

save_dir = pred_results[0].save_dir
for img_path in glob.glob(f'{save_dir}/*.jpg')[:4]:
    display(IPImage(filename=img_path))

## Step 4 -- Export weights for local + edge use

`best.pt` is what the local `ppe_detection.py` script expects.
The ONNX export is optional extra credit for your "edge deployment" slide --
ONNX runs efficiently on a Raspberry Pi without needing PyTorch installed there.

In [ ]:
best_weights_path = f'runs/detect/ppe_detector/weights/best.pt'
print('Best weights at:', best_weights_path)

# Optional: export to ONNX for the edge-deployment story
model.export(format='onnx')

from google.colab import files
files.download(best_weights_path)
# A file-download prompt should appear in your browser -- save it as
# best.pt into the models/ folder of the project on your laptop.

## Next step

Once `best.pt` is downloaded and placed in `models/best.pt` on your laptop,
move to `src/ppe_detection.py` (next part of this project) to run it live on
your webcam.